# Embedding model evaluation -- incident similarity retrieval

Decides which multilingual embedding model backs the Knowledge Base module's similarity engine
(`app/modules/knowledge_base/`), by retrieval quality **and** practical cost on the constrained
target hardware (Intel i5-6300U, 16GB RAM, no GPU), not benchmarks alone.

**Candidates** (all served locally through Ollama, verified against the live Ollama library and
this machine's installed `ollama` package before use):

| Model | Disk size | Params | Context |
|---|---|---|---|
| `qwen3-embedding:0.6b` | 639 MB | ~596M | 32K tokens |
| `embeddinggemma:300m` | 622 MB | ~308M | 2K tokens |
| `bge-m3` | 1.16 GB | ~567M | 8K tokens |

## This notebook runs the production pipeline, not a simplified copy of it

The embedding model is the **only** experimental variable. Everything else -- preprocessing,
identifier extraction, application scoping, hybrid reference matching, ranking, the result cap --
is imported from `app/modules/knowledge_base/` and executed exactly as production executes it:

| Stage | Production | Here |
|---|---|---|
| what gets embedded | `preprocess_description(description).embedding_text` | *same function* |
| candidate scope | same `Application` only | *same* |
| semantic candidates | `find_nearest(..., limit=MAX_RESULTS)`, cosine | same, in numpy |
| reference candidates | `find_referenced(...)` on cited identifiers | same rule, in numpy |
| ranking / fusion / cap | `rank_candidates(...)` | *same function* |

Only the storage engine differs: pgvector's `<=>` in SQL becomes a normalized dot product in
numpy, which is the same quantity. If the production retrieval policy changes, this notebook
changes with it, because it calls that policy rather than restating it.

**Preprocessing.** Ticket descriptions carry organization-specific identifiers (`F19840051125`,
`INC001010948992`, `COL01000000043331119`, `0048GKF1`, ...). Those are replaced by type
placeholders before embedding -- so `COMMANDE BLOQUEE F72136041225` and
`COMMANDE BLOQUEE F19840051125` no longer look similar merely because both end in an 11-digit
run behind an `F` -- and kept as structured data so exact retrieval can still use them.
Codes that carry meaning (`GE0005`, `idFOP_SUPFON_VIOXSUPF06_WARN`, `SMC3`, job and workflow
names, versions) are deliberately left in the text.

**Hybrid retrieval.** A ticket whose description cites an incident, commande or prestation
reference is guaranteed a result slot for the ticket that reference resolves to, bypassing the
score threshold. This is not reachable semantically: one corpus query's entire description is
*"Au sujet du ticket INC001010976766"* and the ticket it names ranks 114th / 222nd / 173rd by
cosine across the three models.

**Score threshold.** Cosine scores are not comparable across models, so a single shared threshold
would measure calibration rather than quality (at 0.6: qwen3 returns a mean of 6.6 results out of
7, embeddinggemma 2.9, and embeddinggemma returns *nothing at all* on a third of queries). K is
fixed at 7 for every model because it is model-agnostic; the threshold is **calibrated per model**
to a shared target result count. The winning model's calibrated value is what should replace
`MIN_SIMILARITY_THRESHOLD` in production.

**Methodology at a glance** -- pooled retrieval evaluation, same queries and same human judgments
reused across all three models:

1. Sample 30 query tickets, stratified by each application's share of the corpus.
2. Retrieve each model's candidates through the production pipeline above.
3. Pool the three models' candidates per query into one deduplicated, blinded judgment queue,
   ordered so the pairs that actually move the metrics come first.
4. Judge each (query, candidate) pair once -- binary relevant / not relevant -- reused by every model.
5. Score Precision@K, pool-relative Recall@K, and MRR on the **final** post-threshold results;
   benchmark latency/memory/size separately. No composite score -- quality and cost side by side.

**How to run this notebook -- "Run All" will NOT pause for you, read this first:**

`ipywidgets` buttons don't block cell execution, so a single "Run All" runs straight through the
judgment widget with zero judgments recorded and produces meaningless (empty) metrics at the end.
The actual workflow has three steps:

1. Run every cell top to bottom **up through the "judgment-store" cell** (the cell just above the
   judgment widget) -- this samples/freezes the benchmark, generates and caches embeddings for all
   three models (slow, CPU-only -- expect ~15 minutes the first time on top of any existing cache,
   instant on reruns), runs retrieval, calibrates thresholds, and loads `judgments` from disk.
2. Run the widget cell below it and judge at your own pace. Safe to stop anytime -- each click
   appends to `cache/judgments.jsonl` immediately, resumable across sessions. **Watch the tier-1
   counter: when it reaches 0 the metrics are complete and you can stop.**
3. When ready to see results (even partial), **re-run starting from the "judgment-store" cell**
   (Jupyter: "Run Selected Cell and All Below" on that cell) and continue down through the end.
   Re-running from section 7 alone is NOT enough -- `judgments` is only loaded once, in the
   judgment-store cell, and clicking the widget's buttons never updates that in-memory variable,
   only the file on disk.

All experiment artifacts live under `cache/` (git-ignored, same as `data/` and `storage/` --
regenerable, and may echo raw ticket content).


In [ ]:
from __future__ import annotations

import hashlib
import json
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

sys.path.insert(0, "../..")

# The production pipeline itself. Nothing below reimplements any of this -- see the header table.
from app.modules.knowledge_base.domain.services.description_preprocessor import preprocess_description
from app.modules.knowledge_base.domain.services.similarity_ranking import (
	MAX_RESULTS,
	MIN_SIMILARITY_THRESHOLD,
	ScoredCandidate,
	rank_candidates,
)
from app.scripts.seeding.ticket_data.processed_tickets import load_processed_tickets

RANDOM_SEED = 42

# 30 queries -> ~374 pooled pairs, of which ~200 are the ones that actually move the metrics (see
# section 6's prioritized queue). Sized to the annotation time available, not to statistical
# comfort -- section 12 records what that costs.
N_QUERIES = 30
LATENCY_SAMPLE_SIZE = 30

# K is production's MAX_RESULTS, not a notebook choice -- it is model-agnostic, so every model is
# given exactly the same number of slots to fill.
TOP_K = MAX_RESULTS

# Each model's threshold is calibrated so it returns this many results per query on average.
# Cosine scales differ enough between these models that a shared constant would compare
# calibration instead of retrieval quality; a shared *operating point* compares the thing we
# actually care about. 4 of 7 sits mid-range for all three -- raise it for a more permissive
# pipeline, lower it for a stricter one, and the comparison stays fair either way.
TARGET_RESULT_COUNT = 4.0

MODEL_TAGS = ["qwen3-embedding:0.6b", "embeddinggemma:300m", "bge-m3"]
OLLAMA_HOST = "http://localhost:11434"

# Bumped whenever preprocessing or retrieval semantics change, so a cache written by the previous
# pipeline can never be silently reused by this one.
PIPELINE_VERSION = "hybrid-v2"

CACHE_DIR = Path("cache")
EMBEDDINGS_DIR = CACHE_DIR / "embeddings"
RETRIEVAL_DIR = CACHE_DIR / "retrieval"
BENCHMARK_CORPUS_PATH = CACHE_DIR / "benchmark_corpus.csv"
BENCHMARK_QUERIES_PATH = CACHE_DIR / "benchmark_queries.json"
JUDGMENTS_PATH = CACHE_DIR / "judgments.jsonl"
MODEL_INFO_PATH = CACHE_DIR / "model_info.json"
RESOURCE_BENCHMARK_PATH = CACHE_DIR / "resource_benchmark.json"
THRESHOLDS_PATH = CACHE_DIR / f"calibrated_thresholds_{PIPELINE_VERSION}.json"

for directory in (CACHE_DIR, EMBEDDINGS_DIR, RETRIEVAL_DIR):
	directory.mkdir(parents=True, exist_ok=True)


def text_digest(text: str) -> str:
	"""Identifies the exact string an embedding was produced from, so a cached vector is only
	reused when the text that produced it is byte-identical."""
	return hashlib.sha1(text.encode("utf-8")).hexdigest()[:16]


print(f"Production policy in force: K={TOP_K}, threshold calibrated per model to ~{TARGET_RESULT_COUNT} results/query")
print(f"(production's current placeholder threshold is {MIN_SIMILARITY_THRESHOLD} -- this notebook exists to replace it)")


### What to clear when something changes

Every artifact under `cache/` is derived, but they depend on different things, so "start over" is rarely the right move. Bottom to top, each layer depends on the ones above it:

| Artifact | Depends on | Clear it when |
|---|---|---|
| `benchmark_queries.json` | `RANDOM_SEED`, `N_QUERIES` | You want a different query sample. **This is the only file whose deletion invalidates judgments**, because judgments are keyed by query id. |
| `benchmark_corpus.csv` | `data/processed/*`, preprocessing | Never manually — it is rewritten automatically when it predates the current preprocessing. |
| `embeddings/*.npz` | (model, exact `embedding_text`) | Never manually. Each vector is stored with a digest of the text that produced it, so changing preprocessing re-embeds exactly the texts that changed and reuses the rest. Embedding is deterministic, so a reused vector is identical to a freshly computed one. |
| `retrieval/*__<PIPELINE_VERSION>.json` | embeddings + retrieval logic | Never manually — bump `PIPELINE_VERSION` instead, which sidesteps the whole file. |
| `calibrated_thresholds_<PIPELINE_VERSION>.json` | retrieval candidates, `TARGET_RESULT_COUNT` | You changed `TARGET_RESULT_COUNT` and want it re-solved. |
| `judgments.jsonl` | **ticket content only** | Essentially never. A judgment answers "is ticket C relevant to ticket Q" — a fact about two tickets. Changing the model, the preprocessing or the threshold changes *which pairs you are shown*, never whether a shown pair is relevant. |
| `model_info.json`, `resource_benchmark.json` | the installed Ollama models | You pulled a new build of a model. |

The one genuinely destructive command, for when you want to resample queries:

```bash
rm cache/benchmark_queries.json cache/judgments.jsonl
```

## 1. Load the ticket corpus and run production preprocessing

Reuses `app.scripts.seeding.ticket_data.processed_tickets.load_processed_tickets()` -- the same historical-data loader the ticket seeder itself uses -- rather than re-parsing `data/processed/*.json`.

Each row then gets the two derived columns the pipeline works from, both produced by production's `preprocess_description`: `embedding_text` (what the model actually sees) and `identifiers` (what exact retrieval matches on). `genergy_id` comes along because it is the right-hand side of every reference lookup. The raw `description` is carried unchanged for the judgment UI -- preprocessing never rewrites it.

In [ ]:
def build_corpus() -> pd.DataFrame:
	"""One row per historical ticket. Position in this fixed concatenation (FCI, COLORIS, AERO,
	VIO, in load_processed_tickets()'s order) is its stable id for this experiment -- the source
	files are a static historical export, not a live table, so this id never changes across reruns.

	`embedding_text` and `identifiers` are produced by the production preprocessor, so a ticket is
	represented here exactly as `GenerateSimilarityResultsHandler` would represent it at ingestion.
	"""
	rows = load_processed_tickets()
	records = []
	for i, row in enumerate(rows):
		preprocessed = preprocess_description(row.description)
		records.append({
			"id": i,
			"application": row.application.value,
			"category": row.category.value,
			"title": row.title,
			"description": row.description,
			"embedding_text": preprocessed.embedding_text,
			"identifiers": json.dumps(
				[[identifier.type.value, identifier.value] for identifier in preprocessed.identifiers]
			),
			"genergy_id": row.genergy_id,
			"resolution_notes": row.resolution_notes,
		})
	return pd.DataFrame.from_records(records)


def has_meaningful_description(text: str | None) -> bool:
	return text is not None and text.strip() != ""


corpus_raw = build_corpus()
corpus = corpus_raw[corpus_raw["embedding_text"].apply(has_meaningful_description)].reset_index(drop=True)

print(f"Loaded {len(corpus_raw)} historical tickets, {len(corpus)} eligible (non-empty text after preprocessing).")
print(pd.concat([
	corpus_raw.groupby("application").size().rename("total"),
	corpus.groupby("application").size().rename("eligible"),
], axis=1))

changed = (corpus["embedding_text"] != corpus["description"].str.strip()).sum()
with_identifiers = (corpus["identifiers"] != "[]").sum()
print(f"\nPreprocessing rewrote {changed} of {len(corpus)} descriptions; {with_identifiers} carry at least one identifier.")
print("\nExample:")
example = corpus[corpus["identifiers"] != "[]"].iloc[0]
print(f"  raw       : {example['description'].strip()[:110]!r}")
print(f"  embedded  : {example['embedding_text'][:110]!r}")
print(f"  extracted : {example['identifiers']}")


## 2. Establish the shared evaluation set (frozen benchmark)

Sampled once, then frozen to `cache/benchmark_corpus.csv` / `cache/benchmark_queries.json`. Every later run loads the frozen set instead of resampling, so embeddings, retrieval results, and judgments all stay pinned to the exact same tickets even if `data/processed/*.json` changes later.

In [ ]:
def sample_queries(eligible: pd.DataFrame, n_queries: int, seed: int) -> pd.DataFrame:
	"""Stratified by each application's share of the eligible corpus (not equal-per-app) --
	confirmed: AERO (54 tickets) gets proportionally few queries rather than being padded to match
	the larger applications.

	Deduplicated by `embedding_text`, not by raw description, so none of the scarce n_queries slots
	are wasted judging what the pipeline cannot tell apart. That distinction now matters:
	"COMMANDE BLOQUEE F72136041225" and "COMMANDE BLOQUEE F19840051125" are different raw
	descriptions but the same text after preprocessing, so as queries they would retrieve
	identically and produce duplicate judgment work.

	Duplicate tickets are NOT removed from the retrievable corpus, only from the *query* pool, so
	recurring incidents remain fully findable as candidates.
	"""
	rng = np.random.default_rng(seed)
	app_counts = eligible.groupby("application").size()
	raw_shares = app_counts / app_counts.sum() * n_queries
	shares = raw_shares.round().astype(int)

	diff = n_queries - shares.sum()
	if diff != 0:
		remainders = raw_shares - shares
		order = remainders.sort_values(ascending=diff < 0).index
		for app in order[: abs(diff)]:
			shares[app] += 1 if diff > 0 else -1

	selected_ids: list[int] = []
	for app, quota in shares.items():
		stratum = eligible[eligible["application"] == app]
		deduped = stratum.drop_duplicates(subset="embedding_text", keep="first")
		quota = min(quota, len(deduped))
		chosen = rng.choice(deduped["id"].to_numpy(), size=quota, replace=False)
		selected_ids.extend(int(x) for x in chosen)

	return eligible[eligible["id"].isin(selected_ids)].reset_index(drop=True)


In [ ]:
REQUIRED_COLUMNS = {"embedding_text", "identifiers", "genergy_id"}

if BENCHMARK_QUERIES_PATH.exists():
	query_ids = json.loads(BENCHMARK_QUERIES_PATH.read_text(encoding="utf-8"))["query_ids"]
	print(f"Frozen query set found on disk -- reusing the same {len(query_ids)} queries (not resampling).")
else:
	print("No frozen query set yet -- sampling now. This happens ONCE; delete cache/benchmark_queries.json to resample.")
	query_ids = sample_queries(corpus, N_QUERIES, RANDOM_SEED)["id"].tolist()
	BENCHMARK_QUERIES_PATH.write_text(
		json.dumps({"query_ids": query_ids, "random_seed": RANDOM_SEED, "n_queries": N_QUERIES}, indent=2),
		encoding="utf-8",
	)

# The frozen corpus snapshot is refreshed when it predates the current preprocessing (it would
# otherwise lack embedding_text/identifiers entirely). Refreshing it is safe and does NOT
# invalidate judgments: ticket ids are positions in a static export, so a given id still names the
# same ticket. The *query set* above is what must never be resampled, and it never is.
if BENCHMARK_CORPUS_PATH.exists():
	frozen = pd.read_csv(BENCHMARK_CORPUS_PATH, encoding="utf-8")
	if REQUIRED_COLUMNS.issubset(frozen.columns):
		corpus = frozen
	else:
		print(f"Frozen corpus predates {PIPELINE_VERSION} preprocessing -- rewriting it with the derived columns.")
		corpus.to_csv(BENCHMARK_CORPUS_PATH, index=False, encoding="utf-8")
else:
	corpus.to_csv(BENCHMARK_CORPUS_PATH, index=False, encoding="utf-8")

corpus["identifiers"] = corpus["identifiers"].fillna("[]")
corpus_by_id = corpus.set_index("id")
queries_df = corpus_by_id.loc[query_ids].reset_index()

missing = set(query_ids) - set(corpus["id"])
assert not missing, f"Frozen queries missing from the corpus: {sorted(missing)}"

print(f"\n{len(corpus)} candidates in the retrievable corpus, {len(query_ids)} frozen queries.")
print(pd.concat([
	corpus.groupby("application").size().rename("candidates"),
	queries_df.groupby("application").size().rename("queries"),
], axis=1))
print("\nCandidate search is scoped per application, so the 'candidates' column above is the real "
	"pool each query draws from -- not the full corpus.")


## 3. Ollama setup & model bookkeeping

Records each model's digest, disk size, architecture, and embedding dimension once, so the final comparison is traceable to an exact model build, not just a name.

In [23]:
import ollama

_client = ollama.Client(host=OLLAMA_HOST)


class EmbeddingFailure(RuntimeError):
	"""Raised when Ollama fails to produce an embedding after all retries."""


def _full_tag(model_tag: str) -> str:
	"""`list()`/`ps()` always report a fully-qualified tag (e.g. `bge-m3` -> `bge-m3:latest`) even
	when the model was referenced without one, so lookups against those responses must normalize
	first or an untagged reference like `bge-m3` never matches.
	"""
	return model_tag if ":" in model_tag else f"{model_tag}:latest"


def embed_one(model_tag: str, text: str, *, max_attempts: int = 3, base_delay: float = 1.0) -> tuple[list[float], int]:
	"""Returns (embedding, total_duration_ns) -- the duration is Ollama's own server-side timing,
	used later for latency benchmarking instead of Python-side wall-clock. Retries transient
	failures (model still loading, connection hiccups) with exponential backoff.
	"""
	last_error: Exception | None = None
	for attempt in range(1, max_attempts + 1):
		try:
			response = _client.embed(model=model_tag, input=text)
			return list(response.embeddings[0]), response.total_duration
		except Exception as exc:
			last_error = exc
			if attempt < max_attempts:
				time.sleep(base_delay * (2 ** (attempt - 1)))
	raise EmbeddingFailure(f"Failed to embed with {model_tag!r} after {max_attempts} attempts: {last_error}") from last_error


def get_model_info(model_tag: str) -> dict:
	"""Static bookkeeping captured once per model: digest + disk size from `list`, architecture and
	embedding dimension from `show`.
	"""
	list_entry = next(m for m in _client.list().models if m.model == _full_tag(model_tag))
	show = _client.show(model_tag)
	embedding_dimension = next(
		(v for k, v in show.modelinfo.items() if k.endswith("embedding_length")), None
	)
	max_context_length = next(
		(v for k, v in show.modelinfo.items() if k.endswith("context_length")), None
	)
	return {
		"model_tag": model_tag,
		"digest": list_entry.digest[:19],
		"disk_size_bytes": list_entry.size,
		"family": show.details.family,
		"parameter_size": show.details.parameter_size,
		"quantization_level": show.details.quantization_level,
		"embedding_dimension": embedding_dimension,
		"max_context_length": max_context_length,
		"capabilities": show.capabilities,
	}


In [24]:
model_info: dict = json.loads(MODEL_INFO_PATH.read_text(encoding="utf-8")) if MODEL_INFO_PATH.exists() else {}

for tag in MODEL_TAGS:
	if tag in model_info:
		continue
	model_info[tag] = get_model_info(tag)
	MODEL_INFO_PATH.write_text(json.dumps(model_info, indent=2), encoding="utf-8")  # checkpoint after each model

pd.DataFrame(model_info).T


,model_tag,digest,disk_size_bytes,family,parameter_size,quantization_level,embedding_dimension,max_context_length,capabilities
qwen3-embedding:0.6b,qwen3-embedding:0.6b,ac6da0dfba84a81fdbf,639150858,qwen3,595.78M,Q8_0,1024,32768,"[tools, thinking, embedding]"
embeddinggemma:300m,embeddinggemma:300m,85462619ee721b466c5,621875917,gemma3,307.58M,BF16,768,2048,[embedding]
bge-m3,bge-m3,7907646426070047a77,1157672605,bert,566.70M,F16,1024,8192,[embedding]


## 4. Embedding generation (cached, resumable)

Embeds each ticket's `embedding_text` -- the preprocessed description, never the raw one, never title/metadata/resolution notes. This is byte-for-byte the string production would hand the embedding provider.

Each cached vector is stored with a digest of the text that produced it, so a vector is only reused when that exact text is unchanged. Vectors cached before preprocessing existed stay valid for the ~two-thirds of tickets whose text preprocessing didn't touch, and only the rest are re-embedded.

In [ ]:
def _npz_path(model_tag: str) -> Path:
	return EMBEDDINGS_DIR / f"{model_tag.replace(':', '_')}.npz"


def load_cached_embeddings(model_tag: str, corpus_df: pd.DataFrame) -> dict[int, np.ndarray]:
	"""Returns only the cached vectors still valid for the current `embedding_text`.

	Caches written before preprocessing existed carry no digests. Rather than discarding them
	wholesale, those are validated against the digest of the *raw* description -- which is what
	they were embedded from -- so every ticket preprocessing left alone keeps its vector and only
	the genuinely changed texts are re-embedded.
	"""
	path = _npz_path(model_tag)
	if not path.exists():
		return {}

	data = np.load(path, allow_pickle=False)
	cached = {int(i): data["embeddings"][row] for row, i in enumerate(data["ids"])}

	if "text_digests" in data:
		cached_digests = {int(i): str(d) for i, d in zip(data["ids"], data["text_digests"])}
	else:
		print(f"{model_tag}: cache predates preprocessing -- keeping only vectors whose text is unchanged.")
		cached_digests = {
			int(row.id): text_digest(row.description) for row in corpus_df.itertuples()
		}

	wanted = {int(row.id): text_digest(row.embedding_text) for row in corpus_df.itertuples()}
	return {
		i: vector for i, vector in cached.items()
		if i in wanted and cached_digests.get(i) == wanted[i]
	}


def save_cached_embeddings(model_tag: str, embeddings: dict[int, np.ndarray], corpus_df: pd.DataFrame) -> None:
	texts = corpus_df.set_index("id")["embedding_text"].to_dict()
	ids = np.array(sorted(embeddings))
	matrix = np.stack([embeddings[i] for i in ids]).astype(np.float32)
	digests = np.array([text_digest(texts[int(i)]) for i in ids])
	np.savez_compressed(_npz_path(model_tag), ids=ids, embeddings=matrix, text_digests=digests)


def embed_corpus(model_tag: str, corpus_df: pd.DataFrame, *, checkpoint_every: int = 25) -> dict[int, np.ndarray]:
	"""Embeds every ticket's preprocessed text with `model_tag`, skipping ids whose cached vector
	is still valid, and checkpointing periodically so a crash never loses more than
	`checkpoint_every` items of CPU-bound work.
	"""
	embeddings = load_cached_embeddings(model_tag, corpus_df)
	todo = corpus_df[~corpus_df["id"].isin(embeddings)]
	if todo.empty:
		print(f"{model_tag}: {len(embeddings)} embeddings already cached and current, nothing to do.")
		return embeddings

	print(f"{model_tag}: embedding {len(todo)} of {len(corpus_df)} tickets ({len(embeddings)} reused from cache)...")
	failures: list[int] = []
	for n, row in enumerate(tqdm(list(todo.itertuples()), desc=model_tag), start=1):
		try:
			vector, _ = embed_one(model_tag, row.embedding_text)
			embeddings[row.id] = np.asarray(vector, dtype=np.float32)
		except EmbeddingFailure as exc:
			print(f"  skipped ticket id={row.id}: {exc}")
			failures.append(row.id)
		if n % checkpoint_every == 0:
			save_cached_embeddings(model_tag, embeddings, corpus_df)

	save_cached_embeddings(model_tag, embeddings, corpus_df)
	if failures:
		print(f"{model_tag}: {len(failures)} tickets failed after retries -- ids: {failures}")
	return embeddings


In [26]:
embeddings_by_model: dict[str, dict[int, np.ndarray]] = {}
for tag in MODEL_TAGS:
	embeddings_by_model[tag] = embed_corpus(tag, corpus)


qwen3-embedding:0.6b: 799 embeddings already cached, nothing to do.
embeddinggemma:300m: 799 embeddings already cached, nothing to do.
bge-m3: 799 embeddings already cached, nothing to do.


## 5. Candidate retrieval -- the production pipeline, in numpy

Two candidate sets per query, mirroring the two methods of `SimilaritySearchPort`:

- **semantic** -- `find_nearest`: cosine similarity (normalized dot product, the same quantity as pgvector's `1 - (a <=> b)`), restricted to the query's own `Application`, excluding the query ticket, capped at `MAX_RESULTS`.
- **referenced** -- `find_referenced`: same application and exclusion, but selected by exact identifier match rather than distance. A candidate matches when a reference identifier cited by the query is either that candidate's own `genergy_id`, or an identifier extracted from its description. Only reference families (incident / commande / prestation) qualify -- a shared user CUID or site code is co-occurrence, not aboutness.

Both are stored **pre-threshold**. The threshold is a per-model calibration applied in the next section, and keeping it out of the cached candidate sets is what lets it be recalibrated later without re-judging anything.

In [ ]:
from app.modules.knowledge_base.domain.enums.identifier_type import IdentifierType


def parse_identifiers(raw: str) -> list[tuple[IdentifierType, str]]:
	return [(IdentifierType(t), v) for t, v in json.loads(raw or "[]")]


def reference_identifiers(raw: str) -> set[tuple[IdentifierType, str]]:
	"""Only the families whose exact match is strong enough to guarantee a slot -- `is_reference`
	is the production classification, not a notebook one."""
	return {(t, v) for t, v in parse_identifiers(raw) if t.is_reference}


class RetrievalIndex:
	"""The lookup structures `PgvectorSimilaritySearch` gets from indexed columns: candidates
	grouped by application, plus the two exact-match maps the reference query rides on.
	"""

	def __init__(self, corpus_df: pd.DataFrame, embeddings: dict[int, np.ndarray]) -> None:
		usable = corpus_df[corpus_df["id"].isin(embeddings)]
		self.ids_by_application: dict[str, np.ndarray] = {}
		self.matrix_by_application: dict[str, np.ndarray] = {}
		for application, group in usable.groupby("application"):
			ids = np.array(sorted(group["id"]))
			matrix = np.stack([embeddings[int(i)] for i in ids]).astype(np.float64)
			self.ids_by_application[application] = ids
			self.matrix_by_application[application] = matrix / np.linalg.norm(matrix, axis=1, keepdims=True)

		self.vectors = {
			int(i): embeddings[int(i)] / np.linalg.norm(embeddings[int(i)]) for i in usable["id"]
		}
		self.application_of = usable.set_index("id")["application"].to_dict()

		self.ids_by_genergy_id: dict[str, list[int]] = {}
		self.ids_by_identifier: dict[tuple[IdentifierType, str], list[int]] = {}
		for row in usable.itertuples():
			if isinstance(row.genergy_id, str) and row.genergy_id:
				self.ids_by_genergy_id.setdefault(row.genergy_id, []).append(int(row.id))
			for key in reference_identifiers(row.identifiers):
				self.ids_by_identifier.setdefault(key, []).append(int(row.id))

	def find_nearest(self, query_id: int, limit: int) -> list[tuple[int, float]]:
		"""Mirrors PgvectorSimilaritySearch.find_nearest."""
		application = self.application_of[query_id]
		ids = self.ids_by_application[application]
		scores = self.matrix_by_application[application] @ self.vectors[query_id]
		results: list[tuple[int, float]] = []
		for idx in np.argsort(-scores):
			candidate_id = int(ids[idx])
			if candidate_id == query_id:
				continue
			results.append((candidate_id, float(scores[idx])))
			if len(results) == limit:
				break
		return results

	def find_referenced(self, query_id: int, references: set[tuple[IdentifierType, str]]) -> list[tuple[int, float]]:
		"""Mirrors PgvectorSimilaritySearch.find_referenced: matched by identifier, scored by
		cosine, unbounded -- the ranking policy decides how many survive."""
		if not references:
			return []
		application = self.application_of[query_id]
		matched: set[int] = set()
		for key in references:
			matched.update(self.ids_by_identifier.get(key, []))
			# The genergy_id side: the ticket being referenced does not repeat the reference in
			# its own description, it *is* the ticket carrying that id.
			matched.update(self.ids_by_genergy_id.get(key[1], []))
		matched.discard(query_id)
		results = [
			(candidate_id, float(self.vectors[candidate_id] @ self.vectors[query_id]))
			for candidate_id in matched
			if self.application_of[candidate_id] == application
		]
		return sorted(results, key=lambda pair: -pair[1])


def retrieve_for_model(
	model_tag: str, embeddings: dict[int, np.ndarray], query_ids: list[int], corpus_df: pd.DataFrame
) -> dict[int, dict[str, list[tuple[int, float]]]]:
	"""Pre-threshold candidate sets per query. Cached per (model, pipeline version)."""
	cache_path = RETRIEVAL_DIR / f"{model_tag.replace(':', '_')}__{PIPELINE_VERSION}.json"
	if cache_path.exists():
		raw = json.loads(cache_path.read_text(encoding="utf-8"))
		return {
			int(qid): {kind: [(int(cid), float(s)) for cid, s in pairs] for kind, pairs in sets.items()}
			for qid, sets in raw.items()
		}

	index = RetrievalIndex(corpus_df, embeddings)
	references_by_id = corpus_df.set_index("id")["identifiers"].to_dict()
	results = {
		qid: {
			"semantic": index.find_nearest(qid, limit=TOP_K),
			"referenced": index.find_referenced(qid, reference_identifiers(references_by_id[qid])),
		}
		for qid in query_ids
	}
	cache_path.write_text(json.dumps({str(qid): sets for qid, sets in results.items()}), encoding="utf-8")
	return results


def apply_policy(
	candidate_sets: dict[str, list[tuple[int, float]]], threshold: float
) -> list[tuple[int, float, bool]]:
	"""Hands the candidates to production's own `rank_candidates` and returns
	(ticket_id, score, matched_reference) in final rank order."""
	ranked = rank_candidates(
		semantic=[ScoredCandidate(ticket_id=cid, similarity_score=s) for cid, s in candidate_sets["semantic"]],
		referenced=[ScoredCandidate(ticket_id=cid, similarity_score=s) for cid, s in candidate_sets["referenced"]],
		min_similarity_score=threshold,
		max_results=TOP_K,
	)
	return [(c.ticket_id, c.similarity_score, c.matched_reference) for c in ranked]


In [ ]:
candidates_by_model: dict[str, dict[int, dict[str, list[tuple[int, float]]]]] = {}
for tag in MODEL_TAGS:
	candidates_by_model[tag] = retrieve_for_model(tag, embeddings_by_model[tag], query_ids, corpus)
	sets = candidates_by_model[tag]
	with_reference = sum(1 for qid in query_ids if sets[qid]["referenced"])
	print(f"{tag}: candidates ready for {len(sets)} queries "
		f"({with_reference} with at least one exact reference match).")

# The reference path is identical across models by construction -- it depends on extracted
# identifiers, not on vectors -- so this is a property of the corpus, not of any model.
reference_hits = {
	qid: candidates_by_model[MODEL_TAGS[0]][qid]["referenced"] for qid in query_ids
}
reference_hits = {qid: hits for qid, hits in reference_hits.items() if hits}
print(f"\nQueries reaching a ticket by exact reference: {len(reference_hits)} / {len(query_ids)}")
for qid, hits in list(reference_hits.items())[:5]:
	best_semantic = candidates_by_model[MODEL_TAGS[0]][qid]["semantic"]
	worst_kept = best_semantic[-1][1] if best_semantic else float("nan")
	for cid, score in hits[:2]:
		reachable = "also in the semantic top-K" if cid in {c for c, _ in best_semantic} else "UNREACHABLE semantically"
		print(f"  #{qid} -> #{cid}: cosine={score:.3f} (semantic top-{TOP_K} floor={worst_kept:.3f}) -- {reachable}")


## 5b. Per-model threshold calibration

`MIN_SIMILARITY_THRESHOLD` is the one production parameter that cannot be shared across models. Cosine scores from these three models live on different scales, so a single constant sets a different operating point for each one — it would compare calibration, not retrieval quality.

Each model's threshold is instead solved so that the **full pipeline returns `TARGET_RESULT_COUNT` results per query on average**. That is a purely distributional calibration: it uses no relevance judgments, so there is no way for it to tune itself on the same data the metrics are scored against.

The table below also reports what the shared 0.6 placeholder would have done to each model, which is the discrepancy this section exists to remove. **The calibrated threshold of the winning model is the value that should replace `MIN_SIMILARITY_THRESHOLD` in `similarity_ranking.py`.**

In [ ]:
def pipeline_results(candidate_sets: dict[int, dict], threshold: float) -> dict[int, list[tuple[int, float, bool]]]:
	"""The final, production-shaped result list for every query at this threshold."""
	return {qid: apply_policy(candidate_sets[qid], threshold) for qid in query_ids}


def mean_result_count(candidate_sets: dict[int, dict], threshold: float) -> float:
	results = pipeline_results(candidate_sets, threshold)
	return float(np.mean([len(results[qid]) for qid in query_ids]))


def empty_query_count(candidate_sets: dict[int, dict], threshold: float) -> int:
	results = pipeline_results(candidate_sets, threshold)
	return sum(1 for qid in query_ids if not results[qid])


def calibrate_threshold(candidate_sets: dict[int, dict], target: float, tolerance: float = 1e-4) -> float:
	"""Solves for the threshold that yields `target` results per query on average.

	Bisection rather than a closed form because the count is a step function of the threshold and
	is also floored by guaranteed reference matches, which ignore the threshold entirely. The
	count is monotonically non-increasing in the threshold, so bisection is well-defined.
	"""
	if mean_result_count(candidate_sets, 0.0) <= target:
		return 0.0
	low, high = 0.0, 1.0
	while high - low > tolerance:
		middle = (low + high) / 2
		if mean_result_count(candidate_sets, middle) >= target:
			low = middle
		else:
			high = middle
	return round(low, 4)


thresholds = json.loads(THRESHOLDS_PATH.read_text(encoding="utf-8")) if THRESHOLDS_PATH.exists() else {}
for tag in MODEL_TAGS:
	if tag not in thresholds:
		thresholds[tag] = calibrate_threshold(candidates_by_model[tag], TARGET_RESULT_COUNT)
		THRESHOLDS_PATH.write_text(json.dumps(thresholds, indent=2), encoding="utf-8")

calibration_df = pd.DataFrame([
	{
		"model": tag,
		"calibrated_threshold": thresholds[tag],
		"mean_results": mean_result_count(candidates_by_model[tag], thresholds[tag]),
		"empty_queries": empty_query_count(candidates_by_model[tag], thresholds[tag]),
		"mean_results_at_0_6": mean_result_count(candidates_by_model[tag], MIN_SIMILARITY_THRESHOLD),
		"empty_queries_at_0_6": empty_query_count(candidates_by_model[tag], MIN_SIMILARITY_THRESHOLD),
	}
	for tag in MODEL_TAGS
]).set_index("model")

# The final results every downstream section scores, judges and inspects.
results_by_model = {tag: pipeline_results(candidates_by_model[tag], thresholds[tag]) for tag in MODEL_TAGS}

print(f"Target: {TARGET_RESULT_COUNT} results per query, K={TOP_K}\n")
print(f"The '_at_0_6' columns show what the shared placeholder threshold ({MIN_SIMILARITY_THRESHOLD}) would")
print("have done to each model -- the unfair comparison this calibration replaces.\n")
calibration_df


## 6. Human relevance judgment (pooled, blinded, reusable)

Each model's candidates per query are pooled into one deduplicated queue; the evaluator judges each (query, candidate) pair exactly once and never sees which model(s) retrieved it — blinding falls out of the pooling design for free. Binary relevant / not relevant, per the earlier decision.

**Pooling is done on the pre-threshold candidate sets**, deliberately wider than what any model actually returns, because a judgment is a statement about two tickets and not about a pipeline configuration. Judging the wider pool means the threshold can be recalibrated — or `TARGET_RESULT_COUNT` changed outright — without invalidating a single recorded judgment.

**The queue is split into two tiers so a short session still produces complete numbers.** Tier 1 is every pair some model actually returns at its calibrated threshold: these are the only pairs Precision and MRR read, so once tier 1 is done the metrics for this configuration are final. Tier 2 is everything the threshold cut — optional work that buys the freedom to re-calibrate later. The widget shows how many tier-1 pairs are left.

In [ ]:
def load_judgments() -> dict[tuple[int, int], bool]:
	if not JUDGMENTS_PATH.exists():
		return {}
	judgments: dict[tuple[int, int], bool] = {}
	with JUDGMENTS_PATH.open(encoding="utf-8") as fh:
		for line in fh:
			line = line.strip()
			if not line:
				continue
			record = json.loads(line)
			judgments[(record["query_id"], record["candidate_id"])] = record["relevant"]
	return judgments


def append_judgment(query_id: int, candidate_id: int, relevant: bool) -> None:
	"""Appends immediately (JSONL, not a rewritten file) so a judging session can be interrupted at
	any point without losing prior work.
	"""
	record = {"query_id": query_id, "candidate_id": candidate_id, "relevant": relevant, "judged_at": time.time()}
	with JUDGMENTS_PATH.open("a", encoding="utf-8") as fh:
		fh.write(json.dumps(record) + "\n")


def build_pooled_pairs(
	candidates_by_model: dict[str, dict[int, dict[str, list[tuple[int, float]]]]],
	results_by_model: dict[str, dict[int, list[tuple[int, float, bool]]]],
	query_ids: list[int],
) -> tuple[list[tuple[int, int]], int]:
	"""Union of every model's pre-threshold candidates (semantic and referenced) per query,
	deduplicated, ordered so the pairs that matter most are judged first.

	Two tiers. Tier 1 is every pair some model actually *returns* at its calibrated threshold:
	these are the only pairs Precision and MRR read, so once tier 1 is judged the metrics for this
	configuration are complete and nothing is understated. Tier 2 is the rest of the pool -- pairs
	a stricter threshold cut, or that only one model surfaced. Judging those buys the freedom to
	recalibrate `TARGET_RESULT_COUNT` later without re-judging, and improves pool-relative recall,
	but it is optional work.

	Within tier 1, pairs a model ranked highly come first, so an interrupted session still covers
	the results a user would most likely have looked at.

	Returns (ordered_pairs, tier_1_size).
	"""
	best_rank: dict[tuple[int, int], int] = {}
	for results in results_by_model.values():
		for qid in query_ids:
			for rank, (cid, _, _) in enumerate(results[qid], start=1):
				key = (qid, cid)
				best_rank[key] = min(best_rank.get(key, rank), rank)

	pool: set[tuple[int, int]] = set()
	for qid in query_ids:
		for model_sets in candidates_by_model.values():
			for kind in ("semantic", "referenced"):
				pool.update((qid, cid) for cid, _ in model_sets[qid][kind])

	returned = sorted(
		(pair for pair in pool if pair in best_rank),
		key=lambda pair: (best_rank[pair], pair[0], pair[1]),
	)
	remainder = sorted(pair for pair in pool if pair not in best_rank)
	return returned + remainder, len(returned)


judgments = load_judgments()
pooled_pairs, tier_1_size = build_pooled_pairs(candidates_by_model, results_by_model, query_ids)
remaining_pairs = [pair for pair in pooled_pairs if pair not in judgments]

tier_1_pairs = set(pooled_pairs[:tier_1_size])
tier_1_done = sum(1 for pair in tier_1_pairs if pair in judgments)

print(f"Pooled (query, candidate) pairs across all {len(MODEL_TAGS)} models: {len(pooled_pairs)}")
print(f"  tier 1 -- actually returned at the calibrated thresholds : {tier_1_size}"
	f"  ({tier_1_done} judged, {tier_1_size - tier_1_done} to go)")
print(f"  tier 2 -- cut by the threshold, optional                 : {len(pooled_pairs) - tier_1_size}")
print()
print("The queue below is ordered tier 1 first, best-ranked first. Judging just tier 1 gives")
print("complete Precision/MRR for this configuration -- tier 2 only buys the ability to change")
print("TARGET_RESULT_COUNT later without re-judging.")
print()
print(f"Already judged: {len(judgments)} | Remaining in queue: {len(remaining_pairs)}")


In [ ]:
import ipywidgets as widgets
from IPython.display import clear_output, display

_judge_state = {"index": 0, "pairs": remaining_pairs}
_tier_1_remaining = [pair for pair in remaining_pairs if pair in tier_1_pairs]


def _render_pair(query_id: int, candidate_id: int) -> str:
	"""Shows the RAW descriptions, never the preprocessed `embedding_text`. You are judging
	whether two tickets are about the same problem -- a question about the tickets, not about what
	the pipeline chose to embed. Showing the placeholders here would leak pipeline behaviour into
	the ground truth.

	Query resolution notes are shown for annotator context only -- a real query ticket at
	inference time is freshly created and has no resolution yet, and notes are never embedded or
	otherwise fed to the retrieval pipeline. They just help judge relevance when the description
	alone is ambiguous (e.g. two tickets resolved the same way are almost certainly relevant to
	each other even if worded differently).
	"""
	q = corpus_by_id.loc[query_id]
	c = corpus_by_id.loc[candidate_id]
	query_resolution = q["resolution_notes"] if pd.notna(q["resolution_notes"]) else "(none recorded)"
	candidate_resolution = c["resolution_notes"] if pd.notna(c["resolution_notes"]) else "(none recorded)"
	return (
		f"<h4>Query ticket #{query_id} -- {q['application']} / {q['category']}</h4>"
		f"<p><b>{q['title']}</b></p><p>{q['description']}</p>"
		f"<p><i>Resolution notes (context only -- real queries won't have this yet):</i> {query_resolution}</p>"
		"<hr>"
		f"<h4>Candidate ticket #{candidate_id} -- {c['application']} / {c['category']}</h4>"
		f"<p><b>{c['title']}</b></p><p>{c['description']}</p>"
		f"<p><i>Resolution notes:</i> {candidate_resolution}</p>"
	)


_output = widgets.Output()
_progress_label = widgets.Label()
_relevant_btn = widgets.Button(description="Relevant", button_style="success")
_not_relevant_btn = widgets.Button(description="Not relevant", button_style="danger")
_skip_btn = widgets.Button(description="Skip for now")


def _show_current() -> None:
	pairs = _judge_state["pairs"]
	idx = _judge_state["index"]
	tier_1_left = sum(1 for pair in pairs[idx:] if pair in tier_1_pairs)
	if tier_1_left:
		status = f"TIER 1 (required): {tier_1_left} left -- metrics are complete once this hits 0"
	else:
		status = "Tier 1 done -- metrics are complete. Everything below is optional."
	_progress_label.value = f"{idx} judged this session | {len(pairs) - idx} in queue | {status}"
	with _output:
		clear_output(wait=True)
		if idx >= len(pairs):
			print("All pooled pairs judged. If you add another model later, re-run the pooling cell "
				"above and this cell to pick up only the newly-added pairs.")
			return
		qid, cid = pairs[idx]
		display(widgets.HTML(_render_pair(qid, cid)))


def _on_relevant(_) -> None:
	qid, cid = _judge_state["pairs"][_judge_state["index"]]
	append_judgment(qid, cid, True)
	_judge_state["index"] += 1
	_show_current()


def _on_not_relevant(_) -> None:
	qid, cid = _judge_state["pairs"][_judge_state["index"]]
	append_judgment(qid, cid, False)
	_judge_state["index"] += 1
	_show_current()


def _on_skip(_) -> None:
	pairs = _judge_state["pairs"]
	pairs.append(pairs.pop(_judge_state["index"]))
	_show_current()


_relevant_btn.on_click(_on_relevant)
_not_relevant_btn.on_click(_on_not_relevant)
_skip_btn.on_click(_on_skip)

display(_progress_label, _output, widgets.HBox([_relevant_btn, _not_relevant_btn, _skip_btn]))
_show_current()


## 7. Retrieval quality metrics

Computed on the **final production-shaped results** — post-preprocessing, post-application-scoping, post-hybrid-fusion, post-threshold, post-cap. These are exactly the tickets a user would have been shown.

Because the threshold is calibrated per model to the same target result count, `K` is no longer a constant length: it is `MAX_RESULTS` in the pipeline but each model returns a variable number of results below it, as production does. Precision is therefore reported over the results actually returned. Recall@K stays *pool-relative* (relative to the union of relevant items any model found for a query), not true corpus-wide recall — see the caveat in the docstring below.

In [ ]:
def compute_metrics(
	results_by_model: dict[str, dict[int, list[tuple[int, float, bool]]]],
	judgments: dict[tuple[int, int], bool],
	query_ids: list[int],
) -> pd.DataFrame:
	"""Precision@K and MRR are well-defined per (model, query) regardless of pooling -- they only
	depend on judgments for that model's own returned items, which are always in the pool by
	construction. Recall@K is necessarily *pool-relative*: relative to the union of relevant items
	ANY model found for that query, not true corpus-wide recall (which would require exhaustively
	judging the whole corpus -- exactly what pooling exists to avoid).

	The pool used for recall is the judged pre-threshold pool, not each model's returned set, so a
	model is correctly penalized for thresholding away a relevant ticket another model returned.

	Queries where the pool contains zero relevant candidates are excluded from the recall average
	(undefined, not zero) but still contribute to precision and MRR, both of which are correctly 0
	in that case. A query that returns nothing at all scores 0 precision and 0 MRR -- returning
	nothing is a real outcome in production, not a missing measurement.
	"""
	pool_relevant_by_query: dict[int, set[int]] = {qid: set() for qid in query_ids}
	for (qid, cid), relevant in judgments.items():
		if relevant and qid in pool_relevant_by_query:
			pool_relevant_by_query[qid].add(cid)

	rows = []
	for model_tag, results in results_by_model.items():
		for qid in query_ids:
			retrieved = [cid for cid, _, _ in results[qid]]
			relevant_retrieved = [cid for cid in retrieved if judgments.get((qid, cid), False)]
			precision = len(relevant_retrieved) / len(retrieved) if retrieved else 0.0
			pool_relevant = pool_relevant_by_query[qid]
			recall = len(relevant_retrieved) / len(pool_relevant) if pool_relevant else float("nan")
			reciprocal_rank = 0.0
			for rank, cid in enumerate(retrieved, start=1):
				if judgments.get((qid, cid), False):
					reciprocal_rank = 1.0 / rank
					break
			rows.append({
				"model": model_tag,
				"query_id": qid,
				"application": corpus_by_id.loc[qid, "application"],
				"n_returned": len(retrieved),
				"n_reference_matches": sum(1 for _, _, is_reference in results[qid] if is_reference),
				"precision_at_k": precision,
				"recall_at_k_pool": recall,
				"reciprocal_rank": reciprocal_rank,
			})
	return pd.DataFrame(rows)


In [ ]:
metrics_df = compute_metrics(results_by_model, judgments, query_ids)

summary = metrics_df.groupby("model").agg(
	precision_at_k=("precision_at_k", "mean"),
	recall_at_k_pool=("recall_at_k_pool", "mean"),
	mrr=("reciprocal_rank", "mean"),
	mean_returned=("n_returned", "mean"),
	n_queries=("query_id", "nunique"),
	n_recall_eligible=("recall_at_k_pool", lambda s: int(s.notna().sum())),
).join(calibration_df[["calibrated_threshold"]])

print(f"Queries with zero relevant candidates found by any model (excluded from recall average): "
	f"{summary['n_queries'].iloc[0] - summary['n_recall_eligible'].iloc[0]} / {len(query_ids)}")
print(f"Judged {len(judgments)} of {len(pooled_pairs)} pooled pairs -- unjudged pairs count as not relevant, "
	f"so partial judging understates every model equally.")
summary


In [ ]:
per_app_summary = metrics_df.groupby(["model", "application"]).agg(
	precision_at_k=("precision_at_k", "mean"),
	recall_at_k_pool=("recall_at_k_pool", "mean"),
	mrr=("reciprocal_rank", "mean"),
	mean_returned=("n_returned", "mean"),
	n=("query_id", "nunique"),
)
print("Per application -- note candidate pools differ in size (AERO 54 vs COLORIS 285), and search")
print("is scoped per application, so these are not directly comparable to each other.\n")
per_app_summary


## 8. Practical performance benchmarking

Cold-start (model load) and warm single-call latency, timed via Ollama's own server-side duration, plus resident memory via `ollama ps` -- all on this exact CPU-only, no-GPU machine, which is the actual constraint we're planning around.

In [ ]:
def sample_latency_texts(corpus_df: pd.DataFrame, n: int, seed: int) -> list[str]:
	"""A fixed sample of real `embedding_text` values, shared across models, so every model is
	timed on identical inputs spanning the corpus's natural length range -- and on the same
	preprocessed strings production would actually send it, not the raw descriptions.
	"""
	rng = np.random.default_rng(seed)
	idx = rng.choice(len(corpus_df), size=min(n, len(corpus_df)), replace=False)
	return corpus_df.iloc[idx]["embedding_text"].tolist()


def benchmark_model(model_tag: str, texts: list[str]) -> dict:
	"""Cold start: force-unloads the model first (keep_alive=0) so the measurement isn't an
	artifact of the model already being resident from the embedding-generation step above, then
	times the reload. Warm latency: repeated single-item calls after that, matching production's
	actual access pattern (GenerateSimilarityResultsHandler embeds exactly one description per
	call, never a batch) -- so single-call latency IS the realistic throughput proxy here, not an
	approximation of it.

	Preprocessing cost is not timed separately: it is pure regex substitution on one short string,
	orders of magnitude below a single embedding call, and it runs identically for every model so
	it cannot shift the comparison.
	"""
	try:
		_client.embed(model=model_tag, input="unload probe", keep_alive=0)
	except Exception:
		pass
	time.sleep(1.0)

	_, cold_duration_ns = embed_one(model_tag, texts[0])

	durations_ms = []
	for text in texts[1:]:
		_, duration_ns = embed_one(model_tag, text)
		durations_ms.append(duration_ns / 1e6)

	running = next((m for m in _client.ps().models if m.model == _full_tag(model_tag)), None)

	return {
		"model_tag": model_tag,
		"cold_start_ms": cold_duration_ns / 1e6,
		"warm_latency_mean_ms": float(np.mean(durations_ms)),
		"warm_latency_median_ms": float(np.median(durations_ms)),
		"warm_latency_p95_ms": float(np.percentile(durations_ms, 95)),
		"approx_throughput_per_sec": 1000.0 / float(np.mean(durations_ms)),
		"resident_memory_mb": (running.size / 1e6) if running else None,
		"resident_memory_vram_mb": (running.size_vram / 1e6) if running else None,
	}


In [ ]:
latency_texts = sample_latency_texts(corpus, LATENCY_SAMPLE_SIZE, seed=RANDOM_SEED)

resource_results: dict = json.loads(RESOURCE_BENCHMARK_PATH.read_text(encoding="utf-8")) if RESOURCE_BENCHMARK_PATH.exists() else {}

for tag in MODEL_TAGS:
	if tag in resource_results:
		continue
	resource_results[tag] = benchmark_model(tag, latency_texts)
	RESOURCE_BENCHMARK_PATH.write_text(json.dumps(resource_results, indent=2), encoding="utf-8")  # checkpoint after each model

resource_df = pd.DataFrame(list(resource_results.values())).assign(
	disk_size_mb=lambda d: d["model_tag"].map(lambda t: model_info[t]["disk_size_bytes"] / 1e6),
)
resource_df


## 9. Embedding space visualization (PCA & UMAP) -- diagnostic only

Not a selection criterion by itself -- used to spot structure or failure patterns that complement the quantitative metrics above. "Nicer-looking" clusters do not imply better retrieval.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA


def plot_projection(projector_name: str, transform_fn) -> None:
	"""Projects the full retrievable corpus (not just the sampled queries) for more visible
	structure; sampled queries are outlined for reference.

	Colouring by application is a diagnostic for the scoping decision, not for retrieval quality:
	search never crosses applications, so clean separation between colours buys nothing. What is
	worth looking at is structure *within* one colour.
	"""
	fig, axes = plt.subplots(1, len(MODEL_TAGS), figsize=(6 * len(MODEL_TAGS), 5))
	apps = sorted(corpus_by_id["application"].unique())
	colors = {app: plt.cm.tab10(i) for i, app in enumerate(apps)}
	for ax, tag in zip(axes, MODEL_TAGS):
		emb = embeddings_by_model[tag]
		ids = np.array(sorted(emb))
		matrix = np.stack([emb[i] for i in ids])
		coords = transform_fn(matrix)
		app_labels = corpus_by_id.loc[ids, "application"]
		for app in apps:
			mask = (app_labels == app).to_numpy()
			ax.scatter(coords[mask, 0], coords[mask, 1], s=8, alpha=0.5, color=colors[app], label=app)
		is_query = np.isin(ids, query_ids)
		ax.scatter(
			coords[is_query, 0], coords[is_query, 1],
			s=40, facecolors="none", edgecolors="black", linewidths=0.8, label="query",
		)
		ax.set_title(f"{projector_name}: {tag}")
		ax.legend(fontsize=7, loc="best")
	plt.tight_layout()
	plt.show()


plot_projection("PCA", lambda m: PCA(n_components=2, random_state=RANDOM_SEED).fit_transform(m))


In [ ]:
import umap

plot_projection(
	"UMAP",
	lambda m: umap.UMAP(n_components=2, random_state=RANDOM_SEED, n_neighbors=15, min_dist=0.1).fit_transform(m),
)


## 10. Comparative analysis: quality vs. cost

Side by side, no weighted composite score -- the trade-off is argued in the final write-up (section 12), not collapsed into one number.

In [ ]:
comparison = summary.join(
	resource_df.set_index("model_tag")[[
		"cold_start_ms", "warm_latency_mean_ms", "warm_latency_p95_ms",
		"approx_throughput_per_sec", "resident_memory_mb", "disk_size_mb",
	]]
)
comparison


In [ ]:
quality_cols = [
	("precision_at_k", f"Precision (K={TOP_K}, calibrated)"),
	("recall_at_k_pool", "Recall (pool-relative)"),
	("mrr", "MRR"),
]
fig, axes = plt.subplots(1, len(quality_cols), figsize=(5 * len(quality_cols), 4))
for ax, (col, title) in zip(axes, quality_cols):
	comparison[col].plot(kind="bar", ax=ax, color="#4C72B0")
	ax.set_title(title)
	ax.set_xlabel("")
plt.tight_layout()
plt.show()

cost_cols = [
	("warm_latency_mean_ms", "Warm latency (ms/call)"),
	("resident_memory_mb", "Resident memory (MB)"),
	("disk_size_mb", "Disk size (MB)"),
]
fig, axes = plt.subplots(1, len(cost_cols), figsize=(5 * len(cost_cols), 4))
for ax, (col, title) in zip(axes, cost_cols):
	comparison[col].plot(kind="bar", ax=ax, color="#DD8452")
	ax.set_title(title)
	ax.set_xlabel("")
plt.tight_layout()
plt.show()


## 11. Failure cases & model disagreement

Queries where models diverge most, for qualitative inspection -- not scoring.

In [ ]:
def show_disagreement_cases(n: int = 5) -> None:
	"""Surfaces queries where models disagree most on precision -- the largest max-min spread --
	for qualitative reading. Useful for spotting systematic failure modes (e.g. one model
	consistently missing short FCI-style descriptions).

	Both the raw and the embedded text are printed, because a disagreement is often explained by
	what preprocessing removed. A `*` marks a result that entered via an exact reference match
	rather than semantically.
	"""
	spread = (
		metrics_df.pivot(index="query_id", columns="model", values="precision_at_k")
		.assign(spread=lambda d: d.max(axis=1) - d.min(axis=1))
		.sort_values("spread", ascending=False)
	)
	for qid in spread.head(n).index:
		q = corpus_by_id.loc[qid]
		print(f"=== Query #{qid} [{q['application']}] precision spread={spread.loc[qid, 'spread']:.2f} ===")
		print(f"  raw      : {q['description'][:160]}")
		print(f"  embedded : {q['embedding_text'][:160]}")
		if q["identifiers"] != "[]":
			print(f"  extracted: {q['identifiers']}")
		for tag in MODEL_TAGS:
			top3 = results_by_model[tag][qid][:3]
			if not top3:
				print(f"  {tag}: (returned nothing at threshold {thresholds[tag]})")
				continue
			parts = []
			for cid, score, is_reference in top3:
				mark = "+" if judgments.get((qid, cid), False) else "-"
				parts.append(f"#{cid}({score:.2f}){mark}{'*' if is_reference else ''}")
			print(f"  {tag}: {', '.join(parts)}")
		print()


show_disagreement_cases()


## 12. Decision

Fill in after running the evaluation (embeddings generated for all three models, tier-1 judgments complete).

**Quality ranking** (Precision / pool-relative Recall / MRR, overall):
_..._

**Cost ranking** (latency, resident memory, disk size) on this machine:
_..._

**Is the quality gap large enough to justify the more expensive model?**
_..._

**Recommendation:**
_..._

**Production changes this decision implies** — the notebook produces these, they are not guesses:

- `MIN_SIMILARITY_THRESHOLD` in `app/modules/knowledge_base/domain/services/similarity_ranking.py`
  → the winning model's `calibrated_threshold` from section 5b. The value currently there (0.6) is
  an unvalidated placeholder and is wrong for at least two of the three candidates.
- A real `EmbeddingProvider` implementation replacing `UnimplementedEmbeddingProvider`, reporting
  the chosen model's `model_name` / `model_version`.
- A follow-up migration pinning the `embedding` column to the winning model's dimension
  (1024 for qwen3-embedding / bge-m3, 768 for embeddinggemma) and adding the real HNSW/IVFFlat
  index, which the first KB migration deliberately left out.

**Known limitations of this evaluation** (carry these into the decision — don't treat the numbers
as more precise than they are):

- **30 queries is a small sample, chosen to fit the annotation time available.** Treat the overall
  ranking as directional; a gap of a few points between two models is not resolvable at this size.
- **Per-application results are not usable.** The stratified sample gives AERO 2 queries and VIO 7.
  Read the overall table, not `per_app_summary`, which is reported for inspection only.
- Single annotator, binary relevance — no inter-annotator agreement measure.
- Recall is pool-relative, not true corpus recall. If only tier 1 was judged, recall is relative to
  the returned pool specifically, which narrows it further.
- **Hybrid retrieval cannot differentiate the models here.** The reference-match path depends on
  extracted identifiers, not on vectors, so it returns identical candidates for all three. It is
  in the pipeline because it is correct for production — it recovers follow-up tickets no
  embedding can reach — but it contributes nothing to the model comparison. Do not read a model
  difference into it.
- The threshold comparison rests on a chosen operating point (`TARGET_RESULT_COUNT`). A different
  target could reorder models on precision-vs-recall; re-run section 5b onwards to check. That is
  cheap, and needs no re-judging if tier 2 was also judged.
- Resource numbers are specific to this CPU-only dev machine and Ollama's local scheduling; they
  may not transfer exactly to wherever this ends up running in production.
- Description-only representation, per the fixed scope of this experiment — title/metadata/
  category are not considered here even though they exist on every ticket.
- Queries retrieve from the whole corpus, including tickets created after them. Production can
  only ever match what was already ingested, so these numbers are mildly optimistic in a way that
  applies equally to all three models.
